# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# NFL workspace readiness
**Purpose:** review actual synchronization and data-readiness receipts before any model training.

This notebook reads local receipts and raw-file metadata already collected by `nfl_workspace.py`. It does not fetch Git, download data, fit models, or change your canonical notebooks. Charts remain empty when receipts are missing; no synthetic values are substituted.

Run the terminal **sync → doctor → data-audit** steps in `START_HERE.md`, then choose the existing Python 3.11 kernel and **Run → Run All Cells**. A passing raw readiness check covers file hashes and schemas, not a complete row-level data audit or model acceptance.


In [ ]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown
pio.renderers.default = 'plotly_mimetype'
EVIDENCE = Path(os.environ.get('NFL_NOTEBOOK_EVIDENCE_ROOT', str(Path.home() / 'nfl-workspace-evidence')))
FIGURES = []
def receipt(name):
    path = EVIDENCE / ('latest_' + name + '.json')
    return json.loads(path.read_text()) if path.is_file() else {}
def present(fig):
    FIGURES.append(fig)
    fig.show()
def offline_report(filename, title):
    destination = EVIDENCE / 'notebook_views' / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    body = '<!doctype html><html><head><meta charset="utf-8"><title>'+title+'</title></head><body><h1>'+title+'</h1>'
    for i, figure in enumerate(FIGURES):
        body += figure.to_html(full_html=False, include_plotlyjs=True if i == 0 else False)
    body += '</body></html>'
    destination.write_text(body)
    print('Saved offline, private HTML:', destination)
    return str(destination)


In [ ]:
sync = receipt('sync')
audit = receipt('data-audit')
environment = receipt('doctor')
fixture = any(r.get('_synthetic_fixture') for r in (sync, audit, environment))
if fixture:
    display(Markdown('## SYNTHETIC SOFTWARE-TEST FIXTURE — NOT YOUR AWS STATE'))
checks = pd.DataFrame([
    {'Stage':'Git source', 'Status':sync.get('status','NOT RUN'), 'Recorded UTC':sync.get('utc','')},
    {'Stage':'Python environment', 'Status':environment.get('status','NOT RUN'), 'Recorded UTC':environment.get('utc','')},
    {'Stage':'Raw inputs and labels', 'Status':audit.get('status','NOT RUN'), 'Recorded UTC':audit.get('utc','')},
])
display(checks)
if sync:
    display(pd.DataFrame([{'Local HEAD':sync.get('head'), 'Remote at read-back':sync.get('remote_main_at_readback'),
        'Raw files unchanged by SHA256':sync.get('raw_files_sha256_verified_unchanged'),
        'Private artifact metadata unchanged':sync.get('private_artifact_metadata_unchanged')}]))
print('Scientific fits in this milestone: 0. New leaderboard score: none.')


## Weekly file coverage
All 18 recorded 2023-season input/output pairs must be present. Missing labels are not replaced with test samples or Analytics-track files. A local file hash is a preservation record; only a verified snapshot checksum establishes equality with that snapshot.


In [ ]:
weeks = pd.DataFrame(audit.get('weeks', []))
if not weeks.empty:
    coverage = np.vstack([weeks['input_present'].astype(int), weeks['output_present'].astype(int)])
    fig = go.Figure(go.Heatmap(z=coverage, x=weeks['week'], y=['Observed input', 'Future labels'],
                              zmin=0, zmax=1, colorbar={'title':'Present'},
                              hovertemplate='Week %{x}<br>%{y}<br>Present: %{z}<extra></extra>'))
    fig.update_layout(title='Raw-file coverage by week — 1 present, 0 missing', xaxis_title='2023 NFL week', height=340)
    present(fig)
else:
    print('No data-audit receipt. Run the data-audit terminal command first.')


In [ ]:
if not weeks.empty:
    sizes = weeks[['week','input_bytes','output_bytes']].melt(id_vars='week',var_name='File type',value_name='Bytes')
    sizes['MiB'] = sizes['Bytes'] / 1024**2
    fig = px.bar(sizes,x='week',y='MiB',color='File type',barmode='group',
                 title='Raw file sizes — storage inventory, not sample counts')
    fig.update_layout(xaxis_title='2023 NFL week',height=410)
    present(fig)
    display(pd.DataFrame([{'Total files':audit.get('file_count'),
                           'Total MiB':audit.get('total_bytes',0)/1024**2,
                           'Organizer API files':audit.get('organizer_api_files')}]))
    for problem in audit.get('problems',[]):
        print('ACTION:', problem)


## Stored notebook outputs in the canonical repository
This inventories saved execution counts and error outputs. An execution count is not proof of source/data lineage or current predictive accuracy. The helper never re-executes historical training notebooks merely to refresh their appearance.


In [ ]:
REPO = Path(sync.get('repo', str(Path.home() / 'nfl-player-trajectory')))
records=[]
for path in sorted((REPO/'notebooks').glob('*.ipynb')):
    book=json.loads(path.read_text())
    c=[x for x in book.get('cells',[]) if x.get('cell_type')=='code']
    records.append({'Notebook':path.name,'Code cells':len(c),
        'Saved executed cells':sum(x.get('execution_count') is not None for x in c),
        'Stored errors':sum(o.get('output_type')=='error' for x in c for o in x.get('outputs',[]))})
notebooks=pd.DataFrame(records)
if not notebooks.empty:
    display(notebooks)
    fig=px.bar(notebooks,x='Notebook',y=['Code cells','Saved executed cells'],barmode='group',
               title='Canonical notebook execution inventory — historical saved outputs')
    fig.update_layout(height=440,xaxis_tickangle=-20)
    present(fig)
else:
    print('Canonical notebook folder is not available in this runtime.')


In [ ]:
ready = (sync.get('status') == 'source_synced_data_preserved'
         and environment.get('status') == 'environment_ready'
         and audit.get('status') == 'raw_readiness_passed')
display(Markdown('## ' + ('Workspace prerequisites passed' if ready else 'Workspace prerequisites are not all verified')))
print('This is not authorization for a large training run. Next: close the existing run, then the 32-play observed-feature smoke.')
if FIGURES:
    html_path=offline_report('workspace_readiness.html', 'NFL workspace readiness' + (' — SYNTHETIC TEST FIXTURE' if fixture else ''))
